# Gộp order_items và order_payments về mức đơn hàng

Task #26 (Story #4): `olist_order_items_dataset` và `olist_order_payments_dataset` có khóa chính kép (composite PK), nhiều dòng trên mỗi `order_id` — phải aggregate trước khi join vào `olist_orders_dataset`, nếu không sẽ fan-out sai số dòng.

Với payments: đã xác nhận qua dữ liệu thật chỉ 2,98% đơn có nhiều hơn 1 dòng thanh toán, nhưng 76% trong số đó dùng từ 2 hình thức thanh toán khác nhau trở lên. Vì hình thức thanh toán là đặc trưng rủi ro được nêu trong `docs/business-processes.md`, chọn phương án giữ nguyên đầy đủ thông tin (không rút gọn về một "phương thức chính") — xem chi tiết ở phần aggregate payments bên dưới.

In [1]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")

orders = pd.read_csv(RAW_DIR / "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv")
order_payments = pd.read_csv(RAW_DIR / "olist_order_payments_dataset.csv")

print(f"orders: {len(orders)} dòng")
print(f"order_items: {len(order_items)} dòng, {order_items['order_id'].nunique()} order_id khác nhau")
print(f"order_payments: {len(order_payments)} dòng, {order_payments['order_id'].nunique()} order_id khác nhau")

orders: 99441 dòng
order_items: 112650 dòng, 98666 order_id khác nhau
order_payments: 103886 dòng, 99440 order_id khác nhau


## Aggregate order_items theo order_id

Mỗi đơn có thể có nhiều sản phẩm/nhiều seller — tính số lượng sản phẩm, số seller khác nhau, tổng giá và tổng phí vận chuyển.

In [2]:
items_agg = order_items.groupby("order_id").agg(
    items_num_items=("order_item_id", "count"),
    items_num_products=("product_id", "nunique"),
    items_num_sellers=("seller_id", "nunique"),
    items_total_price=("price", "sum"),
    items_total_freight=("freight_value", "sum"),
).reset_index()

print(f"{len(items_agg)} order_id sau khi aggregate (đúng bằng số order_id khác nhau ở trên)")
items_agg.head()

98666 order_id sau khi aggregate (đúng bằng số order_id khác nhau ở trên)


,order_id,items_num_items,items_num_products,items_num_sellers,items_total_price,items_total_freight
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,18.14


## Aggregate order_payments theo order_id — giữ nguyên đầy đủ thông tin (phương án C)

Không chọn một "phương thức thanh toán chính" duy nhất. Thay vào đó:
- Tổng số tiền, số dòng thanh toán, số hình thức thanh toán khác nhau, số kỳ trả góp tối đa.
- Tổng tiền theo từng hình thức thanh toán (pivot), và cờ nhị phân có dùng hình thức đó hay không.

Việc rút gọn về 1 giá trị (nếu cần) sẽ để lại cho bước feature engineering ở Story #5/#6.

In [3]:
payments_summary = order_payments.groupby("order_id").agg(
    payment_total_value=("payment_value", "sum"),
    payment_num_rows=("payment_value", "count"),
    payment_num_types=("payment_type", "nunique"),
    payment_max_installments=("payment_installments", "max"),
).reset_index()

payments_by_type = order_payments.pivot_table(
    index="order_id", columns="payment_type", values="payment_value",
    aggfunc="sum", fill_value=0,
)
payments_by_type.columns = [f"payment_value_{c}" for c in payments_by_type.columns]
payments_by_type = payments_by_type.reset_index()

payments_agg = payments_summary.merge(payments_by_type, on="order_id", how="left")

for col in payments_by_type.columns:
    if col == "order_id":
        continue
    flag_col = col.replace("payment_value_", "payment_has_")
    payments_agg[flag_col] = payments_agg[col] > 0

print(f"{len(payments_agg)} order_id sau khi aggregate")
payments_agg.head()

99440 order_id sau khi aggregate


,order_id,payment_total_value,payment_num_rows,payment_num_types,payment_max_installments,payment_value_boleto,payment_value_credit_card,payment_value_debit_card,payment_value_not_defined,payment_value_voucher,payment_has_boleto,payment_has_credit_card,payment_has_debit_card,payment_has_not_defined,payment_has_voucher
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,1,2,0.0,72.19,0.0,0.0,0.0,False,True,False,False,False
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,1,3,0.0,259.83,0.0,0.0,0.0,False,True,False,False,False
2,000229ec398224ef6ca0657da4fc703e,216.87,1,1,5,0.0,216.87,0.0,0.0,0.0,False,True,False,False,False
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,1,2,0.0,25.78,0.0,0.0,0.0,False,True,False,False,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,1,3,0.0,218.04,0.0,0.0,0.0,False,True,False,False,False


## Join vào orders và kiểm tra không bị fan-out

In [4]:
result = orders.merge(items_agg, on="order_id", how="left").merge(payments_agg, on="order_id", how="left")

assert len(result) == len(orders), "Số dòng thay đổi sau khi join — có fan-out!"
assert result["order_id"].is_unique, "order_id không còn unique sau khi join!"
print(f"OK: {len(result)} dòng, bằng đúng số dòng orders gốc ({len(orders)}), order_id vẫn unique.")

missing_items = result["items_num_items"].isna().sum()
missing_payments = result["payment_total_value"].isna().sum()
print(f"Đơn không có dòng nào trong order_items: {missing_items} ({missing_items / len(result):.2%})")
print(f"Đơn không có dòng nào trong order_payments: {missing_payments} ({missing_payments / len(result):.2%})")

OK: 99441 dòng, bằng đúng số dòng orders gốc (99441), order_id vẫn unique.
Đơn không có dòng nào trong order_items: 775 (0.78%)
Đơn không có dòng nào trong order_payments: 1 (0.00%)


## Lưu dataset trung gian

Kết quả Task #26 lưu tạm ở `data/processed/`, Task #27 sẽ đọc lại file này để bổ sung thông tin khách hàng/người bán/sản phẩm/đánh giá.

In [5]:
OUT_PATH = Path("../data/processed/orders_step1_items_payments.csv")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(OUT_PATH, index=False)
print(f"Đã lưu {len(result)} dòng, {len(result.columns)} cột vào {OUT_PATH}")

Đã lưu 99441 dòng, 27 cột vào ..\data\processed\orders_step1_items_payments.csv
